# Movie Preprocess Frame Sampler

`data/movie_preprocess` に保存された前後分割済み mp4 の全フレームを `data/movie_preprocess_frames` に展開し、そこから front/rear を同じフレーム番号でランダム抽出して画像保存します。保存画像はフォルダ分けせず、ファイル名の先頭に `normal_outdoor_front` などの情報を付与します。

## 0. 設定

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import random
import re
import shutil
import cv2
import numpy as np
import pandas as pd
from IPython.display import display


MOVIE_PREPROCESS_DIR = Path("../data/movie_preprocess")
MOVIE_PREPROCESS_FRAMES_DIR = Path("../data/movie_preprocess_frames")
FRAME_EXTRACTION_MANIFEST_PATH = MOVIE_PREPROCESS_FRAMES_DIR / "movie_preprocess_frames_manifest.csv"
OUTPUT_IMAGE_DIR = Path("../data/movie_preprocess_random_frames")
OUTPUT_MANIFEST_PATH = OUTPUT_IMAGE_DIR / "sampled_frame_pairs_manifest.csv"

# These counts are frame-pair counts. Each sampled frame-pair writes one front image and one rear image.
NORMAL_SAMPLE_FRAME_COUNT = 200
ANOMARY_SAMPLE_FRAME_COUNT = 200

# Ratio of sampled frame-pairs to take from indoor videos within each category.
# If indoor videos are unavailable, the shortage is filled from outdoor videos when possible.
NORMAL_INDOOR_SAMPLE_RATIO = 0.0
ANOMARY_INDOOR_SAMPLE_RATIO = 0.0

RANDOM_SEED = 42
DEFAULT_ENVIRONMENT = "outdoor"
ENVIRONMENT_NAMES = ("indoor", "outdoor")
IMAGE_EXTENSION = ".png"
PNG_COMPRESSION = 3
EXTRACT_FRAME_OVERWRITE = False
OUTPUT_IMAGE_OVERWRITE = True

CATEGORY_SAMPLE_SETTINGS = {
    "normal": {
        "sample_frame_count": NORMAL_SAMPLE_FRAME_COUNT,
        "indoor_sample_ratio": NORMAL_INDOOR_SAMPLE_RATIO,
    },
    "anomary": {
        "sample_frame_count": ANOMARY_SAMPLE_FRAME_COUNT,
        "indoor_sample_ratio": ANOMARY_INDOOR_SAMPLE_RATIO,
    },
}


## 1. front/rear 動画ペアの検出

In [ ]:
def read_video_metadata(video_path: Path) -> dict[str, Any]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    metadata = {
        "frame_count": int(capture.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": float(capture.get(cv2.CAP_PROP_FPS)),
        "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    capture.release()
    return metadata


def infer_environment(video_path: Path, category_dir: Path) -> str:
    relative_parts = video_path.relative_to(category_dir).parts
    for part in relative_parts:
        if part in ENVIRONMENT_NAMES:
            return part
    return DEFAULT_ENVIRONMENT


def strip_view_suffix(stem: str, view_name: str) -> str:
    suffix = f"_{view_name}"
    if stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def build_matching_rear_path(front_path: Path, category_dir: Path) -> Path | None:
    relative_parts = list(front_path.relative_to(category_dir).parts)
    try:
        front_part_index = relative_parts.index("front")
    except ValueError:
        return None

    relative_parts[front_part_index] = "rear"
    if front_path.stem.endswith("_front"):
        rear_filename = f"{strip_view_suffix(front_path.stem, 'front')}_rear{front_path.suffix}"
    else:
        rear_filename = front_path.name.replace("front", "rear", 1)
    relative_parts[-1] = rear_filename
    return category_dir.joinpath(*relative_parts)


def discover_front_rear_video_pairs(
    movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    skipped_rows: list[dict[str, Any]] = []

    for category in ["normal", "anomary"]:
        category_dir = movie_preprocess_dir / category
        if not category_dir.exists():
            skipped_rows.append({
                "category": category,
                "reason": "category_dir_missing",
                "path": str(category_dir),
            })
            continue

        front_paths = sorted(
            path
            for path in category_dir.rglob("*.mp4")
            if path.is_file() and "front" in path.relative_to(category_dir).parts
        )

        for front_path in front_paths:
            rear_path = build_matching_rear_path(front_path, category_dir)
            if rear_path is None or not rear_path.exists():
                skipped_rows.append({
                    "category": category,
                    "reason": "rear_pair_missing",
                    "path": str(front_path),
                    "expected_rear_path": str(rear_path) if rear_path is not None else None,
                })
                continue

            front_metadata = read_video_metadata(front_path)
            rear_metadata = read_video_metadata(rear_path)
            frame_count = min(front_metadata["frame_count"], rear_metadata["frame_count"])
            if frame_count <= 0:
                skipped_rows.append({
                    "category": category,
                    "reason": "empty_video_pair",
                    "path": str(front_path),
                    "rear_path": str(rear_path),
                })
                continue

            rows.append({
                "category": category,
                "environment": infer_environment(front_path, category_dir),
                "source_stem": strip_view_suffix(front_path.stem, "front"),
                "front_path": str(front_path),
                "rear_path": str(rear_path),
                "frame_count": int(frame_count),
                "front_frame_count": int(front_metadata["frame_count"]),
                "rear_frame_count": int(rear_metadata["frame_count"]),
                "fps": float(front_metadata["fps"]),
                "width": int(front_metadata["width"]),
                "height": int(front_metadata["height"]),
            })

    pairs_df = pd.DataFrame(rows)
    if not pairs_df.empty:
        pairs_df.insert(0, "pair_id", range(len(pairs_df)))

    skipped_df = pd.DataFrame(skipped_rows)
    print("detected front/rear pair count:", len(pairs_df))
    if not pairs_df.empty:
        print("frame-pair counts by category/environment:")
        print(pairs_df.groupby(["category", "environment"])["frame_count"].sum().to_dict())
    if not skipped_df.empty:
        print("skipped video entries:", len(skipped_df))
        display(skipped_df.head(20))
    return pairs_df


front_rear_pairs_df = discover_front_rear_video_pairs()
display(front_rear_pairs_df.head())


## 2. 全フレーム画像展開とランダム抽出

In [ ]:
def sanitize_filename_part(value: str) -> str:
    sanitized = re.sub(r"[\\/:*?\"<>|\s]+", "_", str(value)).strip("._")
    return sanitized or "video"


def infer_video_view(video_path: Path, movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR) -> str | None:
    relative_parts = video_path.relative_to(movie_preprocess_dir).parts
    for part in relative_parts:
        if part in {"front", "rear"}:
            return part
    if video_path.stem.endswith("_front"):
        return "front"
    if video_path.stem.endswith("_rear"):
        return "rear"
    return None


def build_frame_image_path(
    frame_image_dir: Path,
    category: str,
    environment: str,
    view_name: str,
    source_stem: str,
    frame_index: int,
) -> Path:
    filename = (
        f"{category}_{environment}_{view_name}_"
        f"{sanitize_filename_part(source_stem)}_"
        f"frame{frame_index:06d}{IMAGE_EXTENSION}"
    )
    return frame_image_dir / filename


def write_image_bgr(image_path: Path, frame_bgr: np.ndarray) -> None:
    image_path.parent.mkdir(parents=True, exist_ok=True)
    if IMAGE_EXTENSION.lower() == ".png":
        params = [cv2.IMWRITE_PNG_COMPRESSION, int(PNG_COMPRESSION)]
    else:
        params = []
    success = cv2.imwrite(str(image_path), frame_bgr, params)
    if not success:
        raise RuntimeError(f"Failed to write image: {image_path}")


def discover_movie_preprocess_videos(
    movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    skipped_rows: list[dict[str, Any]] = []
    for video_path in sorted(movie_preprocess_dir.rglob("*.mp4")):
        if not video_path.is_file():
            continue
        relative_parts = video_path.relative_to(movie_preprocess_dir).parts
        if not relative_parts:
            continue
        category = relative_parts[0]
        category_dir = movie_preprocess_dir / category
        environment = infer_environment(video_path, category_dir)
        view_name = infer_video_view(video_path, movie_preprocess_dir)
        if view_name is None:
            skipped_rows.append({
                "video_path": str(video_path),
                "reason": "view_not_inferred",
            })
            continue
        metadata = read_video_metadata(video_path)
        rows.append({
            "category": category,
            "environment": environment,
            "view": view_name,
            "source_stem": strip_view_suffix(video_path.stem, view_name),
            "video_path": str(video_path),
            "frame_count": int(metadata["frame_count"]),
            "fps": float(metadata["fps"]),
            "width": int(metadata["width"]),
            "height": int(metadata["height"]),
        })

    videos_df = pd.DataFrame(rows)
    if not videos_df.empty:
        videos_df.insert(0, "video_id", range(len(videos_df)))
    skipped_df = pd.DataFrame(skipped_rows)
    print("movie preprocess video count:", len(videos_df))
    if not skipped_df.empty:
        print("skipped movie preprocess videos:", len(skipped_df))
        display(skipped_df.head(20))
    return videos_df


def expected_frame_image_paths(video: pd.Series, frame_image_dir: Path) -> list[Path]:
    return [
        build_frame_image_path(
            frame_image_dir,
            str(video["category"]),
            str(video["environment"]),
            str(video["view"]),
            str(video["source_stem"]),
            frame_index,
        )
        for frame_index in range(int(video["frame_count"]))
    ]


def extract_video_frames_to_images(
    video: pd.Series,
    frame_image_dir: Path = MOVIE_PREPROCESS_FRAMES_DIR,
    overwrite: bool = EXTRACT_FRAME_OVERWRITE,
) -> dict[str, Any]:
    video_path = Path(video["video_path"])
    expected_paths = expected_frame_image_paths(video, frame_image_dir)
    if expected_paths and not overwrite and all(path.exists() for path in expected_paths):
        return {
            "video_id": int(video["video_id"]),
            "video_path": str(video_path),
            "category": video["category"],
            "environment": video["environment"],
            "view": video["view"],
            "source_stem": video["source_stem"],
            "status": "skipped_existing",
            "metadata_frame_count": int(video["frame_count"]),
            "read_frame_count": 0,
            "written_frame_count": 0,
            "existing_frame_count": int(video["frame_count"]),
            "error": None,
        }

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        return {
            "video_id": int(video["video_id"]),
            "video_path": str(video_path),
            "category": video["category"],
            "environment": video["environment"],
            "view": video["view"],
            "source_stem": video["source_stem"],
            "status": "error",
            "metadata_frame_count": int(video["frame_count"]),
            "read_frame_count": 0,
            "written_frame_count": 0,
            "existing_frame_count": 0,
            "error": f"Failed to open video: {video_path}",
        }

    frame_index = 0
    written_frame_count = 0
    existing_frame_count = 0
    error = None
    try:
        while capture.isOpened():
            success, frame_bgr = capture.read()
            if not success:
                break
            frame_image_path = build_frame_image_path(
                frame_image_dir,
                str(video["category"]),
                str(video["environment"]),
                str(video["view"]),
                str(video["source_stem"]),
                frame_index,
            )
            if frame_image_path.exists() and not overwrite:
                existing_frame_count += 1
            else:
                write_image_bgr(frame_image_path, frame_bgr)
                written_frame_count += 1
            frame_index += 1
    except Exception as exception:
        error = repr(exception)
    finally:
        capture.release()

    if error is not None:
        status = "error"
    elif written_frame_count > 0:
        status = "written"
    else:
        status = "skipped_existing"

    return {
        "video_id": int(video["video_id"]),
        "video_path": str(video_path),
        "category": video["category"],
        "environment": video["environment"],
        "view": video["view"],
        "source_stem": video["source_stem"],
        "status": status,
        "metadata_frame_count": int(video["frame_count"]),
        "read_frame_count": int(frame_index),
        "written_frame_count": int(written_frame_count),
        "existing_frame_count": int(existing_frame_count),
        "error": error,
    }


def extract_all_movie_preprocess_frames(
    videos_df: pd.DataFrame,
    frame_image_dir: Path = MOVIE_PREPROCESS_FRAMES_DIR,
    manifest_path: Path = FRAME_EXTRACTION_MANIFEST_PATH,
    overwrite: bool = EXTRACT_FRAME_OVERWRITE,
) -> pd.DataFrame:
    frame_image_dir.mkdir(parents=True, exist_ok=True)
    rows: list[dict[str, Any]] = []
    for video_number, (_, video) in enumerate(videos_df.iterrows(), start=1):
        print(
            f"[{video_number}/{len(videos_df)}] extract frames:",
            video["category"],
            video["environment"],
            video["view"],
            video["source_stem"],
        )
        result = extract_video_frames_to_images(video, frame_image_dir=frame_image_dir, overwrite=overwrite)
        rows.append(result)
        print(
            "  status:", result["status"],
            "read:", result["read_frame_count"],
            "written:", result["written_frame_count"],
            "existing:", result["existing_frame_count"],
        )

    summary_df = pd.DataFrame(rows)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(manifest_path, index=False)
    print("frame extraction manifest saved to:", manifest_path)
    if not summary_df.empty:
        print("frame extraction status counts:")
        print(summary_df["status"].value_counts(dropna=False).to_dict())
    return summary_df


movie_preprocess_videos_df = discover_movie_preprocess_videos()
frame_extraction_summary_df = extract_all_movie_preprocess_frames(movie_preprocess_videos_df)
display(frame_extraction_summary_df.head())


def split_environment_targets(total_count: int, indoor_ratio: float) -> dict[str, int]:
    total_count = max(int(total_count), 0)
    indoor_ratio = float(np.clip(indoor_ratio, 0.0, 1.0))
    indoor_count = int(round(total_count * indoor_ratio))
    return {
        "indoor": indoor_count,
        "outdoor": total_count - indoor_count,
    }


def sample_frame_candidates(
    pairs_df: pd.DataFrame,
    requested_count: int,
    rng: random.Random,
    excluded_keys: set[tuple[int, int]] | None = None,
) -> list[tuple[int, int]]:
    if requested_count <= 0 or pairs_df.empty:
        return []

    excluded_keys = excluded_keys or set()
    candidates: list[tuple[int, int]] = []
    for _, pair in pairs_df.iterrows():
        pair_id = int(pair["pair_id"])
        category = str(pair["category"])
        environment = str(pair["environment"])
        source_stem = str(pair["source_stem"])
        for frame_index in range(int(pair["frame_count"])):
            key = (pair_id, frame_index)
            if key in excluded_keys:
                continue
            front_frame_image_path = build_frame_image_path(
                MOVIE_PREPROCESS_FRAMES_DIR,
                category,
                environment,
                "front",
                source_stem,
                frame_index,
            )
            rear_frame_image_path = build_frame_image_path(
                MOVIE_PREPROCESS_FRAMES_DIR,
                category,
                environment,
                "rear",
                source_stem,
                frame_index,
            )
            if front_frame_image_path.exists() and rear_frame_image_path.exists():
                candidates.append(key)

    if requested_count >= len(candidates):
        return candidates
    return rng.sample(candidates, requested_count)


def build_category_sampling_plan(
    pairs_df: pd.DataFrame,
    category: str,
    sample_frame_count: int,
    indoor_sample_ratio: float,
    rng: random.Random,
) -> list[dict[str, Any]]:
    category_pairs_df = pairs_df[pairs_df["category"] == category]
    environment_targets = split_environment_targets(sample_frame_count, indoor_sample_ratio)

    selected_keys: list[tuple[int, int]] = []
    selected_key_set: set[tuple[int, int]] = set()
    requested_total = sum(environment_targets.values())

    for environment, target_count in environment_targets.items():
        environment_pairs_df = category_pairs_df[category_pairs_df["environment"] == environment]
        environment_keys = sample_frame_candidates(
            environment_pairs_df,
            target_count,
            rng,
            excluded_keys=selected_key_set,
        )
        selected_keys.extend(environment_keys)
        selected_key_set.update(environment_keys)

    shortage_count = requested_total - len(selected_keys)
    if shortage_count > 0:
        top_up_keys = sample_frame_candidates(
            category_pairs_df,
            shortage_count,
            rng,
            excluded_keys=selected_key_set,
        )
        selected_keys.extend(top_up_keys)
        selected_key_set.update(top_up_keys)

    rng.shuffle(selected_keys)
    pair_lookup = pairs_df.set_index("pair_id")
    records: list[dict[str, Any]] = []
    for pair_id, frame_index in selected_keys:
        pair = pair_lookup.loc[pair_id]
        records.append({
            "category": category,
            "environment": pair["environment"],
            "pair_id": int(pair_id),
            "source_stem": pair["source_stem"],
            "frame_index": int(frame_index),
        })

    print(
        "sampling plan:",
        category,
        "requested=", requested_total,
        "planned=", len(records),
        "targets=", environment_targets,
    )
    return records


def build_sampling_plan(
    pairs_df: pd.DataFrame,
    category_sample_settings: dict[str, dict[str, float | int]] = CATEGORY_SAMPLE_SETTINGS,
    random_seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    if pairs_df.empty:
        return pd.DataFrame()

    rng = random.Random(random_seed)
    records: list[dict[str, Any]] = []
    for category, settings in category_sample_settings.items():
        records.extend(build_category_sampling_plan(
            pairs_df=pairs_df,
            category=category,
            sample_frame_count=int(settings["sample_frame_count"]),
            indoor_sample_ratio=float(settings["indoor_sample_ratio"]),
            rng=rng,
        ))

    sampling_plan_df = pd.DataFrame(records)
    if not sampling_plan_df.empty:
        sampling_plan_df.insert(0, "sample_index", range(1, len(sampling_plan_df) + 1))
    return sampling_plan_df
def build_output_image_path(
    output_image_dir: Path,
    sample_index: int,
    category: str,
    environment: str,
    view_name: str,
    source_stem: str,
    frame_index: int,
) -> Path:
    filename = (
        f"{category}_{environment}_{view_name}_"
        f"{sample_index:06d}_{sanitize_filename_part(source_stem)}_"
        f"frame{frame_index:06d}{IMAGE_EXTENSION}"
    )
    return output_image_dir / filename


def copy_sampled_frame_image(source_image_path: Path, output_image_path: Path, overwrite: bool) -> str:
    if not source_image_path.exists():
        raise FileNotFoundError(f"Source frame image does not exist: {source_image_path}")
    if output_image_path.exists() and not overwrite:
        return "skipped_existing"
    output_image_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_image_path, output_image_path)
    return "written"


def save_sampled_frame_pairs(
    sampling_plan_df: pd.DataFrame,
    pairs_df: pd.DataFrame,
    output_image_dir: Path = OUTPUT_IMAGE_DIR,
    overwrite: bool = OUTPUT_IMAGE_OVERWRITE,
) -> pd.DataFrame:
    output_image_dir.mkdir(parents=True, exist_ok=True)
    pair_lookup = pairs_df.set_index("pair_id")
    manifest_rows: list[dict[str, Any]] = []

    for _, sample in sampling_plan_df.iterrows():
        sample_index = int(sample["sample_index"])
        pair_id = int(sample["pair_id"])
        frame_index = int(sample["frame_index"])
        pair = pair_lookup.loc[pair_id]
        category = str(pair["category"])
        environment = str(pair["environment"])
        source_stem = str(pair["source_stem"])

        front_output_path = build_output_image_path(
            output_image_dir,
            sample_index,
            category,
            environment,
            "front",
            source_stem,
            frame_index,
        )
        rear_output_path = build_output_image_path(
            output_image_dir,
            sample_index,
            category,
            environment,
            "rear",
            source_stem,
            frame_index,
        )
        front_frame_source_path = build_frame_image_path(
            MOVIE_PREPROCESS_FRAMES_DIR,
            category,
            environment,
            "front",
            source_stem,
            frame_index,
        )
        rear_frame_source_path = build_frame_image_path(
            MOVIE_PREPROCESS_FRAMES_DIR,
            category,
            environment,
            "rear",
            source_stem,
            frame_index,
        )

        status = "written"
        front_copy_status = "not_started"
        rear_copy_status = "not_started"
        error = None
        try:
            front_copy_status = copy_sampled_frame_image(front_frame_source_path, front_output_path, overwrite)
            rear_copy_status = copy_sampled_frame_image(rear_frame_source_path, rear_output_path, overwrite)
            if front_copy_status == "skipped_existing" and rear_copy_status == "skipped_existing":
                status = "skipped_existing"
        except Exception as exception:
            status = "error"
            error = repr(exception)

        manifest_rows.append({
            "sample_index": sample_index,
            "category": category,
            "environment": environment,
            "source_stem": source_stem,
            "pair_id": pair_id,
            "frame_index": frame_index,
            "front_source_path": pair["front_path"],
            "rear_source_path": pair["rear_path"],
            "front_frame_source_path": str(front_frame_source_path),
            "rear_frame_source_path": str(rear_frame_source_path),
            "front_image_path": str(front_output_path),
            "rear_image_path": str(rear_output_path),
            "front_copy_status": front_copy_status,
            "rear_copy_status": rear_copy_status,
            "status": status,
            "error": error,
        })

    return pd.DataFrame(manifest_rows)


sampling_plan_df = build_sampling_plan(front_rear_pairs_df)
print("planned frame-pair sample count:", len(sampling_plan_df))
display(sampling_plan_df.head())

if sampling_plan_df.empty:
    sampled_frame_pairs_manifest_df = pd.DataFrame()
    print("Skip image export because there are no sampled frame-pairs.")
else:
    sampled_frame_pairs_manifest_df = save_sampled_frame_pairs(
        sampling_plan_df,
        front_rear_pairs_df,
    )
    OUTPUT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    sampled_frame_pairs_manifest_df.to_csv(OUTPUT_MANIFEST_PATH, index=False)
    print("saved front/rear image pair count:", len(sampled_frame_pairs_manifest_df))
    print("manifest saved to:", OUTPUT_MANIFEST_PATH)
    print("status counts:")
    print(sampled_frame_pairs_manifest_df["status"].value_counts(dropna=False).to_dict())
    display(sampled_frame_pairs_manifest_df.head())
